In [ ]:
import numpy as np
import pandas as pd 

import statsmodels.api as sm 


import sys
from pathlib import Path

sys.path.append(r"C:\Users\jcmar\my_files\SportsBetting\BettingStrategy\ModelStrategy\LogisticRegression")
from get_train_test import TrainTestBuilder


In [16]:

fp = r'C:\Users\jcmar\my_files\SportsBetting\data\training_data\entire_odds_stats_2026-03-07.csv'
df_model = pd.read_csv(fp)

df_model['math_red'] = df_model['math_red'].astype('category')
df_model['math_blue'] = df_model['math_blue'].astype('category')
df_model['elo_pred'] = df_model['elo_pred'].astype('category')

non_feats = [
    'date','event_date','event_location','fighter_blue','fighter_red',
    'method','og_blue_name','og_red_fighter', 'red_fighter_stats', 'blue_fighter_stats',
    'pimp_close1_blue','pimp_close1_red','pimp_close2_blue','pimp_close2_red',
    'juice_close1_blue','juice_close1_red','juice_close2_blue','juice_close2_red',
    'line_movement_close1_blue','line_movement_close1_red','line_movement_close2_blue','line_movement_close2_red',
    'winner_name',
    
    'red_fighter_odds','blue_fighter_odds',
    'dec_close1_blue','dec_close1_red','dec_close2_blue','dec_close2_red',
    'dec_fair_close1_blue','dec_fair_close1_red','dec_fair_close2_blue','dec_fair_close2_red',
    'red_ud_to_fav_close1','red_ud_to_fav_close2','blue_ud_to_fav_close1','blue_ud_to_fav_close2',
    'red_stayed_fav_close1','red_stayed_fav_close2','blue_stayed_fav_close1','blue_stayed_fav_close2',
    'red_fav_to_ud_close1','red_fav_to_ud_close2','blue_fav_to_ud_close1','blue_fav_to_ud_close2',
    'red_stayed_dog_close1','red_stayed_dog_close2','blue_stayed_dog_close1','blue_stayed_dog_close2',
    'proba_fair_close1_red','proba_fair_close1_blue','proba_fair_close2_red','proba_fair_close2_blue',
    'performance_bonus_winner', 'fight_otn_bonus', 'close1_blue','close1_red', 'close2_blue', 'close2_red'
] 

selected_feats = [
                  'proba_fair_open_diff', 'reach_diff', 
                  
                  'sub_att_pm_red', 'sub_att_pm_blue',
                  'ratio_control_diff',

                  'td_landed_pm_diff',  
                  'ratio_td_diff', 
                  'adjusted_td_red', 'adjusted_td_blue',

                  'sig_str_absorbed_total_diff', 
                  'sig_str_accuracy_pct_diff',
                  'sig_str_defense_pct_diff',
                  'adjusted_sig_str_blue', 'adjusted_sig_str_red', 
                  
                  'win_pct_red', 'win_pct_blue',
                  'win_streak_diff', 'lose_streak_diff',
                  'elo_red', 'elo_blue', 'elo_pred', 'age_red', 'age_blue',
                  ]



In [17]:
y = 'winner'
builder = TrainTestBuilder(df=df_model, target_col=y, non_features=non_feats, train_size=0.85, random_state=42)
builder.filter_by_date(year=2010, month=2, day=26, date_col='event_date')
X_train, X_test, y_train, y_test, df_train, df_test, scaler_open = builder.prepare_train_test(selected_feats)


Filtered: kept 6650 rows from 2010-02-26 onward.
PREPARE SHAPE: (4256, 297)
MODEL SHAPE: (4177, 297)
Categorical columns: ['elo_pred']
Numerical columns: ['proba_fair_open_diff', 'reach_diff', 'sub_att_pm_red', 'sub_att_pm_blue', 'ratio_control_diff', 'td_landed_pm_diff', 'ratio_td_diff', 'adjusted_td_red', 'adjusted_td_blue', 'sig_str_absorbed_total_diff', 'sig_str_accuracy_pct_diff', 'sig_str_defense_pct_diff', 'adjusted_sig_str_blue', 'adjusted_sig_str_red', 'win_pct_red', 'win_pct_blue', 'win_streak_diff', 'lose_streak_diff', 'elo_red', 'elo_blue', 'age_red', 'age_blue']


In [4]:
def InverseProbabilityNC(predicted_score, y):
    """
    Computes nonconformity scores based on inverse probability.
    """
    prob = np.zeros(y.size, dtype=np.float32)
    for i, y_ in enumerate(y):
        if y_ >= predicted_score.shape[1]:
            prob[i] = 0
        else:
            prob[i] = predicted_score[i, int(y_)]
    return 1 - prob


In [ ]:

def compute_tcp_pvalues(X_train, y_train, X_test, n_classes=2):
    """
    Computes Transductive Conformal Prediction p-values for a test set.

    Returns:
        p_values_tcp : np.ndarray of shape (n_test, n_classes)
    """
    # ------------------------------
    # Base training model
    # ------------------------------
    base_model = sm.Logit(y_train, X_train).fit(disp=False)

    p_train = base_model.predict(X_train)
    pred_train = np.column_stack([1 - p_train, p_train])

    # training nonconformity
    nc_train = InverseProbabilityNC(pred_train, y_train.values)

    # ------------------------------
    # Transductive Conformal Prediction
    # ------------------------------
    n_test = len(X_test)
    p_values_tcp = np.zeros((n_test, n_classes))

    for i in range(n_test):

        x_i = X_test.iloc[[i]]  # shape (1,d)

        for c in range(n_classes):

            # Augmented dataset (train + test point)
            X_aug = sm.add_constant(np.vstack([X_train, x_i]))
            y_aug = np.concatenate([y_train.values, np.array([c])])

            # Fit new model
            model_aug = sm.Logit(y_aug, X_aug).fit(disp=False)

            # Predicted probabilities for augmented dataset
            p_aug = model_aug.predict(X_aug)
            pred_aug = np.column_stack([1 - p_aug, p_aug])

            # Compute nonconformity scores
            nc_aug = InverseProbabilityNC(pred_aug, y_aug)

            # TCP p-value for test point
            test_nc = nc_aug[-1]
            calib_nc = nc_aug[:-1]
            p_values_tcp[i, c] = (np.sum(calib_nc >= test_nc) + 1) / (len(calib_nc) + 1)

    return p_values_tcp


cp_values = compute_tcp_pvalues(X_train, y_train, X_test)


In [24]:
def tcp_train_pvalues(X, y, n_classes=2):
    n = len(X)
    pvals = np.zeros((n, n_classes))

    for i in range(n):
        for c in range(n_classes):

            # leave-one-out
            X_loo = X.drop(i)
            y_loo = np.delete(y, i)

            # augment with candidate label
            X_aug = sm.add_constant(pd.concat([X_loo, X.iloc[[i]]]))
            y_aug = np.concatenate([y_loo, [c]])

            # fit model
            model = sm.Logit(y_aug, X_aug).fit(disp=False)

            # predict on augmented data
            p_aug = model.predict(X_aug)

            pred_aug = np.column_stack([1 - p_aug, p_aug])

            # nonconformity
            nc_aug = InverseProbabilityNC(pred_aug, y_aug)

            test_nc = nc_aug[-1]
            calib_nc = nc_aug[:-1]

            pvals[i, c] = (np.sum(calib_nc >= test_nc) + 1) / (len(calib_nc) + 1)

    return pvals

cp_train_vals = tcp_train_pvalues(X_train, y_train)

In [85]:
def subset_cp_vals(cp_values, alpha=.1):

    prediction_sets = (cp_values <= alpha).astype(int) 
    set0 = ((prediction_sets[:,0] == 1) & (prediction_sets[:,1]==0))
    set1 = ((prediction_sets[:,0] == 0) & (prediction_sets[:,1]==1))
    set2 = ((prediction_sets[:,0] == 0) & (prediction_sets[:,1]==0))
    set3 = ((prediction_sets[:,0] == 1) & (prediction_sets[:,1]==1))
    return [set0, set1, set2, set3]

subsets_test = subset_cp_vals(cp_values)
subsets_train = subset_cp_vals(cp_train_vals)

In [106]:
from sklearn.metrics import brier_score_loss
df_test_naive = pd.read_csv(r'C:\Users\jcmar\my_files\SportsBetting\data\model_results\test_logit_open.csv')


In [136]:

def train_test_subset(subset_train, subset_test, X_train, y_train, X_test, y_test, df_naive):

    data = {'naive_acc': [], 'subset_acc': [], 'subset_test_size': [], 'subset_train_size':[], 'model_brier':[], 'naive_brier':[]}

    for i in range(len(subset_train)):

        X_train_sub = X_train.iloc[subset_train[i]]
        y_train_sub = y_train.iloc[subset_train[i]]

        X_test_sub = X_test.iloc[subset_test[i]]
        y_test_sub = y_test.iloc[subset_test[i]]

        if X_train_sub.shape[0] == 0: 
            continue

        print(X_train_sub.shape)
        # add intercept
        X_train_const = sm.add_constant(X_train_sub, has_constant='add')

        # fit model
        model = sm.Logit(y_train_sub, X_train_const).fit_regularized(
        method="l1",     
        alpha=2.35,  
        )

        # IMPORTANT: add constant to test with same structure
        X_test_const = sm.add_constant(X_test_sub, has_constant='add')
        X_test_const = X_test_const[X_train_const.columns]  # enforce same column order

        # predict probabilities
        y_hat = model.predict(X_test_const)

        # classify
        y_hat_class = (y_hat >= 0.5).astype(int)

        # accuracy
        y_hat_acc = np.mean(y_hat_class == y_test_sub.values)
        model_brier = brier_score_loss(y_test_sub, y_hat)

        # naive accuracy
        naive_sub = df_naive.iloc[subset_test[i]]
        naive_acc = np.mean(naive_sub['pred_winner'].values == naive_sub['winner'].values)

        naive_brier = brier_score_loss(y_test_sub , naive_sub['proba_red'].values )
        assert np.all(naive_sub['winner'] == y_test_sub), 'wrong naive'

        data['model_brier'].append(model_brier)
        data['naive_brier'].append(naive_brier)

        data['naive_acc'].append(naive_acc)
        data['subset_test_size'].append(sum(subset_test[i]))
        data['subset_train_size'].append(sum(subset_train[i]))

        data['subset_acc'].append(y_hat_acc)

    print(brier_score_loss(y_test, df_naive['proba_red'].values ))
    return data


data = train_test_subset(subsets_train, subsets_test, X_train, y_train, X_test, y_test, df_test_naive)



(1067, 23)
Optimization terminated successfully    (Exit mode 0)
            Current function value: 0.5340173039865873
            Iterations: 85
            Function evaluations: 86
            Gradient evaluations: 85
(323, 23)
Optimization terminated successfully    (Exit mode 0)
            Current function value: 0.5770502594529876
            Iterations: 59
            Function evaluations: 60
            Gradient evaluations: 59
(2160, 23)
Optimization terminated successfully    (Exit mode 0)
            Current function value: 0.6690066396012031
            Iterations: 78
            Function evaluations: 78
            Gradient evaluations: 78
0.20027393828540715


In [137]:
df_data = pd.DataFrame(data)
df_data.head()

,naive_acc,subset_acc,subset_test_size,subset_train_size,model_brier,naive_brier
0,0.786458,0.776042,192,1067,0.171753,0.163398
1,0.798319,0.789916,119,323,0.170968,0.169215
2,0.636076,0.629747,316,2160,0.231234,0.234376


In [100]:
weights = [192, 119, 316] 
arr = [0.163398, 0.458087, .255]

weighted_avg = np.average(arr, weights=weights)
weighted_avg

np.float64(0.2654940494417863)

In [135]:
import xgboost as xgb
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

X_train_sub = X_train[subsets_train[0]] 
X_test_sub = X_test[subsets_test[0]]

y_train_sub = y_train[subsets_train[0]]
y_test_sub = y_test[subsets_test[0]]

# model
model = xgb.XGBClassifier(
    n_estimators=800,
    max_depth=3,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    use_label_encoder=False
)

# train
print(X_train_sub.shape)
model.fit(X_train_sub, y_train_sub)

# predict
y_pred = model.predict(X_test_sub)

# accuracy
acc = accuracy_score(y_test_sub, y_pred)
print("Accuracy:", acc)

(1067, 23)


c:\Users\jcmar\my_files\SportsBetting\venv\Lib\site-packages\xgboost\core.py:158: UserWarning: [00:53:04] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Accuracy: 0.7708333333333334


In [ ]:
X_test_sub

,proba_fair_open_diff,reach_diff,sub_att_pm_red,sub_att_pm_blue,ratio_control_diff,td_landed_pm_diff,ratio_td_diff,adjusted_td_red,adjusted_td_blue,sig_str_absorbed_total_diff,...,adjusted_sig_str_red,win_pct_red,win_pct_blue,win_streak_diff,lose_streak_diff,elo_red,elo_blue,age_red,age_blue,elo_pred


In [138]:
2e-3

0.002